# Экспорт данных с дашбордов и сборка сводной таблицы

Ноутбук делает следующее:

1. Читает справочник дашбордов (Excel) с колонками `Дэш`, `Экран`, `Ссылка`.
2. Последовательно открывает каждую ссылку в Chrome (через Selenium).
3. Ждёт 20 секунд, чтобы виджеты прогрузились.
4. Эмулирует **Ctrl+A / Ctrl+C** реальными нажатиями клавиш и сохраняет скопированный текст в `txt/<сегодняшняя_дата>/<Дэш>_<Экран>.txt`.
5. Проходит по всем строкам справочника.
6. Разбирает txt-файлы на отдельные виджеты (показатели) и собирает сводный Excel:
   - Лист 1 «Показатели» — Показатель, Дэш, Экран, Ссылка, Актуально, Есть прогноз, Есть недельный срез.
   - Лист 2 «Помесячные значения» — Показатель, Дэш, Экран, Период, Факт, Выполнение плана.

**Установка зависимостей:**
```
pip install pandas openpyxl selenium pyperclip
```
Linux: `sudo apt install xclip` (иначе буфер обмена не работает).

## Как устроен парсер (по реальному примеру)

Скопированный текст на самом деле построчный (по одному значению на строку), просто при показе мне текста переносы иногда "склеиваются". Структура одного виджета такая:

```
Чистая прибыль        <- заголовок
Млн руб                 <- единица измерения (по ней вычисляется, что предыдущая строка - заголовок)
Прогноз / 5.0            <- KPI-плашки (label, потом значение) - не идут в итоговую таблицу,
План / 67)                  но сканируются на ключевые слова "прогноз"/"нед"
Дельта нед. / -0.1        <- маркер недельного среза (слово "нед")
Факт/прогноз              <- легенда графика (тоже сканируется на "прогноз")
План
Вып.
Авг2025 Сен Окт ... Дек    <- месяцы, ГОД указывается только при смене года
3,4 3,5 3,6 ...             <- числа, по одному на строку, столько же, сколько месяцев
```

⚠️ **Ограничения, о которых стоит знать:**
- Если у виджета вообще нет отдельной строки/чисел плана (как в примере) — колонка «Выполнение плана» будет пустой. Это не баг, а особенность конкретного виджета.
- Разделить в тексте, где заканчивается факт и начинается прогноз внутри объединённой серии «Факт/прогноз», невозможно — поэтому «Актуально» вычисляется как **последний месяц ≤ сегодняшней дате**, а не по самим данным.
- Заголовок виджета определяется как строка, стоящая прямо перед распознанной единицей измерения (`Млн руб`, `Тыс руб`, `%`, `руб`, `шт` и т.п. — список в `UNIT_PATTERNS`, дополните под свои дашборды). Если у какого-то виджета единицы измерения нет или она нестандартная - его заголовок не определится верно. Пришлите мне такой пример, если такое есть.

In [ ]:
import os
import re
import time
from pathlib import Path
from datetime import date

import pandas as pd

# ================== НАСТРОЙКИ ==================
REFERENCE_XLSX = "справочник_дашей.xlsx"
SHEET_NAME = 0
WAIT_SECONDS = 20

OUTPUT_TXT_ROOT = Path("txt")
TODAY_FOLDER = OUTPUT_TXT_ROOT / date.today().isoformat()
RESULT_XLSX = f"Сводная_таблица_{date.today().isoformat()}.xlsx"

CHROME_USER_DATA_DIR = None  # путь к профилю Chrome, если нужна авторизация
CHROME_PROFILE = "Default"

TODAY_FOLDER.mkdir(parents=True, exist_ok=True)
print("Файлы будут сохранены в:", TODAY_FOLDER.resolve())

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains


def create_driver():
    options = webdriver.ChromeOptions()
    if CHROME_USER_DATA_DIR:
        options.add_argument(f"--user-data-dir={CHROME_USER_DATA_DIR}")
        options.add_argument(f"--profile-directory={CHROME_PROFILE}")
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    return driver


driver = create_driver()

In [ ]:
import pyperclip


def copy_page_content(driver, pause=0.4):
    '''Эмулирует выделение всей страницы (Ctrl+A) и копирование (Ctrl+C).'''
    pyperclip.copy("")
    body = driver.find_element(By.TAG_NAME, "body")
    body.click()
    actions = ActionChains(driver)
    actions.key_down(Keys.CONTROL).send_keys("a").key_up(Keys.CONTROL).perform()
    time.sleep(pause)
    actions.key_down(Keys.CONTROL).send_keys("c").key_up(Keys.CONTROL).perform()
    time.sleep(pause)

    text = pyperclip.paste()
    if not text.strip():
        time.sleep(1)
        text = pyperclip.paste()
    return text


def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    return name

## Шаг 1-6. Проход по всем ссылкам и сохранение txt-файлов

In [ ]:
ref_df = pd.read_excel(REFERENCE_XLSX, sheet_name=SHEET_NAME)
ref_df = ref_df.rename(columns=lambda c: str(c).strip())

required_cols = {"Дэш", "Экран", "Ссылка"}
missing = required_cols - set(ref_df.columns)
if missing:
    raise ValueError(f"В справочнике не хватает колонок: {missing}")

errors = []

for i, row in ref_df.iterrows():
    dash = str(row["Дэш"]).strip()
    screen = str(row["Экран"]).strip()
    url = str(row["Ссылка"]).strip()

    fname = f"{safe_filename(dash)}_{safe_filename(screen)}.txt"
    fpath = TODAY_FOLDER / fname

    print(f"[{i + 1}/{len(ref_df)}] {dash} / {screen} -> {url}")

    try:
        driver.get(url)
        time.sleep(WAIT_SECONDS)
        content = copy_page_content(driver)

        if not content.strip():
            print(f"  ⚠️ Пусто! Проверьте вручную: {url}")

        fpath.write_text(content, encoding="utf-8")
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")
        errors.append((dash, screen, url, str(e)))

print("Готово. Файлы сохранены в:", TODAY_FOLDER)
if errors:
    print("Ошибки:")
    for e in errors:
        print(" ", e)

In [ ]:
driver.quit()

## Шаг 7. Парсинг txt-файлов

Настройки ниже (единицы измерения, ключевые слова) можно и нужно дополнять под свои дашборды.

In [ ]:
MONTHS_RU = {
    "янв": 1, "января": 1, "январь": 1,
    "фев": 2, "февраля": 2, "февраль": 2,
    "мар": 3, "марта": 3, "март": 3,
    "апр": 4, "апреля": 4, "апрель": 4,
    "май": 5, "мая": 5,
    "июн": 6, "июня": 6, "июнь": 6,
    "июл": 7, "июля": 7, "июль": 7,
    "авг": 8, "августа": 8, "август": 8,
    "сен": 9, "сентября": 9, "сентябрь": 9,
    "окт": 10, "октября": 10, "октябрь": 10,
    "ноя": 11, "ноября": 11, "ноябрь": 11,
    "дек": 12, "декабря": 12, "декабрь": 12,
}

# месяц с явным годом: "авг2025", "янв.24", "январь 2024", "01.2024", "2024-01"
MONTH_WITH_YEAR_RE = re.compile(
    r"^(?:"
    r"(?P<name>[а-яё]+)\.?\s*['`]?(?P<y1>\d{2,4})"
    r"|(?P<mm>\d{1,2})[./-](?P<y2>\d{2,4})"
    r"|(?P<y3>\d{4})[./-](?P<mm2>\d{1,2})"
    r")$",
    re.IGNORECASE,
)
# месяц без года: "сен", "окт", "январь"
BARE_MONTH_RE = re.compile(r"^[а-яё]+$", re.IGNORECASE)

NUMBER_RE = re.compile(r"^-?\d[\d\s.,]*%?$")

FORECAST_KEYWORDS = ("прогноз",)
WEEKLY_KEYWORDS = ("нед",)

# единицы измерения - по строке с единицей определяется, что предыдущая строка - заголовок
UNIT_PATTERNS = [
    re.compile(r"^(млн|тыс|млрд)\.?\s*руб\.?$", re.IGNORECASE),
    re.compile(r"^руб\.?$", re.IGNORECASE),
    re.compile(r"^%$"),
    re.compile(r"^(млн|тыс|млрд)?\.?\s*шт\.?$", re.IGNORECASE),
    re.compile(r"^(млн|тыс|млрд)?\.?\s*ед\.?$", re.IGNORECASE),
    # добавьте сюда свои варианты единиц измерения, если парсер их не находит
]


def is_unit_line(token: str) -> bool:
    t = token.strip()
    return any(p.match(t) for p in UNIT_PATTERNS)


def _month_name_to_num(name: str):
    name = name.lower()
    for key, num in MONTHS_RU.items():
        if name.startswith(key):
            return num
    return None


def parse_month_token(token: str, current_year):
    '''Возвращает ((год, месяц), новый_текущий_год) либо (None, current_year).'''
    t = token.strip().lower()

    m = MONTH_WITH_YEAR_RE.match(t)
    if m:
        if m.group("name"):
            month = _month_name_to_num(m.group("name"))
            year = m.group("y1")
        elif m.group("mm"):
            month = int(m.group("mm"))
            year = m.group("y2")
        else:
            month = int(m.group("mm2"))
            year = m.group("y3")
        if month is None:
            return None, current_year
        year = int(year)
        if year < 100:
            year += 2000
        if not (1 <= month <= 12):
            return None, current_year
        return (year, month), year

    if BARE_MONTH_RE.match(t) and current_year is not None:
        month = _month_name_to_num(t)
        if month is not None:
            return (current_year, month), current_year

    return None, current_year


def split_line(line: str):
    if "\t" in line:
        parts = line.split("\t")
    else:
        parts = re.split(r"\s{2,}", line)
    return [p.strip() for p in parts if p.strip() != ""]


def to_number(s: str):
    s = s.replace("\xa0", "").replace(" ", "").replace("%", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def flatten_tokens(text: str):
    tokens = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        tokens.extend(split_line(line))
    return tokens

In [ ]:
def parse_widgets(text: str):
    '''
    Разбирает текст, скопированный с экрана, на отдельные виджеты (показатели).
    Возвращает список словарей:
      {"title", "forecast", "weekly", "months": [(год, месяц), ...], "fact": [...], "plan": [...]}
    '''
    tokens = flatten_tokens(text)
    n = len(tokens)
    widgets = []

    pending_meta = []   # строки-кандидаты в заголовок (включая "мусор" типа меню)
    current = None
    current_year = None

    def new_widget(title):
        return {"title": title, "forecast": False, "weekly": False,
                "months": [], "fact": [], "plan": []}

    def scan_keywords(widget, token):
        low = token.lower()
        if any(k in low for k in FORECAST_KEYWORDS):
            widget["forecast"] = True
        if any(k in low for k in WEEKLY_KEYWORDS):
            widget["weekly"] = True

    i = 0
    while i < n:
        tok = tokens[i]

        # 1) строка единицы измерения -> подтверждаем заголовок виджета
        if is_unit_line(tok) and pending_meta:
            if current and current["months"]:
                widgets.append(current)
            current = new_widget(pending_meta[-1])
            current_year = None
            scan_keywords(current, tok)
            pending_meta = []
            i += 1
            continue

        # 2) месяц (с годом или без)
        if current is not None:
            parsed, y2 = parse_month_token(tok, current_year)
        else:
            parsed, y2 = None, current_year

        if parsed is not None:
            months_buf = [parsed]
            current_year = y2
            i += 1
            while i < n:
                p2, y3 = parse_month_token(tokens[i], current_year)
                if p2 is None:
                    break
                months_buf.append(p2)
                current_year = y3
                i += 1
            current["months"] = months_buf

            def consume_numbers(limit):
                nonlocal i
                vals = []
                while i < n and len(vals) < limit and NUMBER_RE.match(tokens[i]):
                    vals.append(to_number(tokens[i]))
                    i += 1
                return vals

            current["fact"] = consume_numbers(len(months_buf))
            current["plan"] = consume_numbers(len(months_buf))
            continue

        # 3) ключевые слова (прогноз / недельный срез) сканируем на лету
        if current is not None:
            scan_keywords(current, tok)

        # 4) обычный текст - кандидат в заголовок следующего виджета
        pending_meta.append(tok)
        i += 1

    if current and current["months"]:
        widgets.append(current)

    return widgets

### Проверка парсера на реальном примере

Ниже - самопроверка на тексте из вашего файла-примера (`Чистая прибыль`). Если результат выглядит правильно - можно переходить к сборке итогового Excel.

In [ ]:
_sample_text = '''Обзор
Монитор бизнеса
Цели
ДИб
БИБ
ХУБ
Чистая прибыль
Млн руб
Прогноз
5.0
План
67)
Дельта нед.
-0.1
Факт/прогноз
План
Вып.
Авг2025
Сен
Окт
Ноя
Дек
Янв2026
Фев
Мар
Апр
Май
Июн
Июл
Авг
Сен
Окт
Ноя
Дек
3,4
3,5
3,6
3.7
3.8
3.9
4.2
4.3
4.4
4.5
4.5
4.6
5.7
5.3
4.7
5.3
5.4
'''

_test_widgets = parse_widgets(_sample_text)
for w in _test_widgets:
    print("Показатель:", w["title"])
    print("  Прогноз:", w["forecast"], "| Недельный срез:", w["weekly"])
    print("  Месяцев:", len(w["months"]), "| Факт значений:", len(w["fact"]), "| План значений:", len(w["plan"]))
    print("  Первый месяц:", w["months"][0], "Последний месяц:", w["months"][-1])

## Шаг 8. Сборка итогового Excel по всем txt-файлам

In [ ]:
def compute_actual_period(months, fact, today=None):
    '''
    'Актуально' = последний месяц <= сегодняшней дате, для которого есть значение факта.
    Это прокси: текстовая копия не позволяет однозначно отличить факт от прогноза
    внутри объединённой серии "Факт/прогноз".
    '''
    today = today or date.today()
    dated = [ym for ym, val in zip(months, fact) if val is not None]
    if not dated:
        return None
    past_or_present = [ym for ym in dated if (ym[0], ym[1]) <= (today.year, today.month)]
    return max(past_or_present) if past_or_present else max(dated)


summary_rows = []
monthly_rows = []

for _, row in ref_df.iterrows():
    dash = str(row["Дэш"]).strip()
    screen = str(row["Экран"]).strip()
    url = str(row["Ссылка"]).strip()
    fname = f"{safe_filename(dash)}_{safe_filename(screen)}.txt"
    fpath = TODAY_FOLDER / fname

    if not fpath.exists():
        print(f"Пропускаю (нет файла): {fpath}")
        continue

    text = fpath.read_text(encoding="utf-8")
    widgets = parse_widgets(text)

    if not widgets:
        print(f"⚠️ Ни одного виджета не распознано в {fpath.name} - проверьте формат файла")

    for w in widgets:
        months = w["months"]
        fact = w["fact"]
        plan = w["plan"]

        if not months or not fact:
            continue

        actual_period = compute_actual_period(months, fact)
        actual_str = f"{actual_period[1]:02d}.{actual_period[0]}" if actual_period else ""

        summary_rows.append({
            "Показатель": w["title"],
            "Дэш": dash,
            "Экран": screen,
            "Ссылка": url,
            "Актуально": actual_str,
            "Есть прогноз": "Да" if w["forecast"] else "Нет",
            "Есть недельный срез": "Да" if w["weekly"] else "Нет",
        })

        # план короче факта -> выравниваем по максимальным (последним) датам
        plan_months = months[len(months) - len(plan):] if plan else []
        plan_dict = dict(zip(plan_months, plan))

        for idx, ym in enumerate(months):
            fact_val = fact[idx] if idx < len(fact) else None
            plan_val = plan_dict.get(ym)
            monthly_rows.append({
                "Показатель": w["title"],
                "Дэш": dash,
                "Экран": screen,
                "Год": ym[0],
                "Месяц": ym[1],
                "Период": f"{ym[1]:02d}.{ym[0]}",
                "Факт": fact_val,
                "Выполнение плана": plan_val,
            })

summary_df = pd.DataFrame(summary_rows)
monthly_df = pd.DataFrame(monthly_rows)

with pd.ExcelWriter(RESULT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Показатели", index=False)
    monthly_df.to_excel(writer, sheet_name="Помесячные значения", index=False)

print("Сохранено:", RESULT_XLSX)
summary_df.head()